In [8]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import src.openmeteo as api
import src.queimada as q
import requests
import time
from datetime import date

In [9]:
df = pd.read_csv('../bdqueimadas.csv')

In [10]:
# Colunas não utilizadas pra treinamento
df = df.drop(columns=['Satelite', 'Pais', 'Estado', 'FRP', 'Latitude', 'Longitude'])

# Apaga os dados dos biomas que não sejam mata atlântica e cerrado
df = df[df["Bioma"].isin(["Mata Atlântica", "Cerrado"])]

# Converte o bioma em um valor numérico baseado no tipo de vegetação.
dados = {'Mata Atlântica': 1.7, 'Cerrado': 3}
df['Bioma'] = df['Bioma'].map(dados)

# Filtra os dados quebrados
df = df[df["DiaSemChuva"] >= 0]
df = df[df["RiscoFogo"] >= 0]

# Garante que a coluna é datetime
df['DataHora'] = pd.to_datetime(df['DataHora'])

# Cria uma coluna temporária com o dia para filtrar as repetidas do mesmo dia (mantendo sua lógica original)
df['Data_Temp'] = df['DataHora'].dt.date
df = df.drop_duplicates(subset=['Data_Temp', 'Municipio'], keep='first')
df = df.drop(columns=['Data_Temp']) # Remove a coluna temporária

# Agora, converte a DataHora para o período mensal (Ex: 2025-09)
df['DataHora'] = df['DataHora'].dt.to_period('M')

# Agrupa por Mês e Município, e tira a média de todas as outras colunas (DiaSemChuva, RiscoFogo, Bioma)
df = df.groupby(['DataHora', 'Municipio']).mean().reset_index()

# Ordena por lugar e data
df = df.sort_values(by=['Municipio', 'DataHora'])
df = df.reset_index(drop=True)

# !APENAS PARA TESTES
df = df.sample(frac=0.05)

In [7]:
# Adiciona a coluna "fogo" = 1 para todos os dados do bdqueimadas (dias do ano em que cada local REGISTROU foco de calor)
df['Fogo'] = 1


# Adiciona as colunas faltantes, onde "fogo" = 0 (dias do ano em que cada local NÃO registrou foco de calor)

municipios = df['Municipio'].unique()
todos_os_meses = (
    pd.date_range(start='2025-6', end='2026-6', freq='MS')
    .strftime('%Y-%m')
    .tolist()
)
df_agrupado = df.groupby('Municipio')['DataHora'].unique()

for municipio in municipios:
    # Meses existentes para este município específico
    meses_existentes = (
        df_agrupado[municipio]
        if municipio in df_agrupado
        else []
    )

    for mes in todos_os_meses:
        if mes not in meses_existentes:
            dia = date.strptime(mes, '%Y-%m')

            resultado = None

            for tentativa in range(10):
                try:
                    resultado = api.obter_dados_clima(dia, municipio)
                    break

                except (requests.exceptions.HTTPError, requests.exceptions.RequestException):
                    print(f'Erro: tentando de novo em {2 ** (tentativa + 1)} segundos')
                    if tentativa == 9:
                        raise
                    time.sleep(2 ** (tentativa + 1))

            if resultado != None:
                vegetacao = df[df['Municipio'] == municipio]['Bioma'].mean()
                umidade = resultado['umidade']
                temperatura = resultado['temperatura']
                precipitacao = resultado['precipitacao']
                precs = {k: v for k, v in resultado.items() if k.startswith('prec')}

                pse = q.periodo_de_secura(precs)
                rf = q.risco_fogo(vegetacao, pse, umidade, temperatura)

                nova_linha = pd.DataFrame(
                    [[mes, municipio, vegetacao, pse, precipitacao, rf, 0]],
                    columns=df.columns
                )

                df = pd.concat([df, nova_linha], ignore_index=True)

            else:
                raise Exception


KeyboardInterrupt: 

In [ ]:
# Número total de municípios únicos
total_municipios = df['Municipio'].nunique()

# Conta quantas vezes cada município aparece e depois tira a média
media_aparicoes = df['Municipio'].value_counts().mean()

print(f'Número de municípios: {total_municipios}')
print(f'Média de focos por município: {media_aparicoes:.2f}')

# Salva o dataframe em um arquivo CSV
df.to_csv("../out.csv", index=False)